<a href="https://colab.research.google.com/github/tayviss/Tayvis_INFO4670_Fall2026/blob/main/INFOR4760_Week4_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

df_students_raw = pd.read_csv('student_records.csv')
df_enrollments_raw = pd.read_csv('course_enrollments.csv')
df_activity_raw = pd.read_csv('weekly_activity.csv')


df_students = df_students_raw.copy()
df_enrollments = df_enrollments_raw.copy()
df_activity = df_activity_raw.copy()

print("Initial shapes:")
print("Students:", df_students.shape)
print("Enrollments:", df_enrollments.shape)
print("Activity:", df_activity.shape)

Initial shapes:
Students: (2027, 12)
Enrollments: (8088, 4)
Activity: (32000, 4)


In [18]:
n_before = len(df_students)
print(f"Sample size before cleaning: {n_before}")

print(df_students.isnull().sum())

median_study_hours = df_students['study_hours_reported'].median()
df_students['study_hours_reported'] = df_students['study_hours_reported'].fillna(median_study_hours)

print("Housing value counts before standardization:")
print(df_students['housing'].value_counts(dropna=False))

housing_mapping = {
    'on-campus': 'On-Campus', 'On campus': 'On-Campus', 'On-Campus': 'On-Campus',
    'off-campus': 'Off-Campus', 'Off campus': 'Off-Campus', 'Off-Campus': 'Off-Campus',
    'commuter': 'Commuter', 'Commuter': 'Commuter',
    'With Family': 'With Family', 'with family': 'With Family', 'With family': 'With Family'
}
df_students['housing'] = df_students['housing'].map(housing_mapping).fillna(df_students['housing'])

print("Housing value counts after standardization:")
print(df_students['housing'].value_counts(dropna=False))

df_students = df_students[(df_students['age'] >= 16) & (df_students['age'] <= 90)]
df_students = df_students[(df_students['work_hours_per_week'] >= 0) & (df_students['work_hours_per_week'] <= 80)]

df_students = df_students.drop_duplicates(subset=['student_id'])

n_after = len(df_students)
print(f"Sample size after cleaning student_records: {n_after}")

Sample size before cleaning: 1990
student_id              0
major                   0
age                     0
housing                 0
commute_miles           0
work_hours_per_week     0
advisor_meetings        0
credits_attempted       0
final_gpa               0
study_hours_reported    0
enrollment_date         0
dropped_out             0
dtype: int64
Housing value counts before standardization:
housing
Off-Campus     743
On-Campus      539
With Family    413
off campus     110
Off-campus      76
with family     69
 On-Campus      40
Name: count, dtype: int64
Housing value counts after standardization:
housing
Off-Campus     743
On-Campus      539
With Family    482
off campus     110
Off-campus      76
 On-Campus      40
Name: count, dtype: int64
Sample size after cleaning student_records: 1990


In [20]:
print("Sample student roster IDs:", df_students['student_id'].head(5).tolist())
print("Sample enrollment IDs:", df_enrollments['student_id'].head(5).tolist())

Sample student roster IDs: ['NU-101508', 'NU-102438', 'NU-102385', 'NU-100767', 'NU-106054']
Sample enrollment IDs: ['108933', '109441', '102076', '106381', '106185']


In [21]:
df_enrollments = df_enrollments.rename(columns={'sid': 'student_id'})
df_enrollments['student_id'] = 'NU-' + df_enrollments['student_id'].astype(str).str.strip()

df_students['student_id'] = df_students['student_id'].astype(str).str.strip()
df_activity['student_id'] = df_activity['student_id'].astype(str).str.strip()

activity_summary = df_activity.groupby('student_id').agg(
    total_minutes_active=('minutes_active', 'sum'),
    total_lms_clicks=('lms_clicks', 'sum')
).reset_index()

enrollment_summary = df_enrollments.groupby('student_id').agg(
    total_courses_enrolled=('course', 'count')
).reset_index()

df_integrated = df_students.merge(enrollment_summary, on='student_id', how='left')
df_integrated = df_integrated.merge(activity_summary, on='student_id', how='left')

matched_rows = df_integrated['total_courses_enrolled'].notnull().sum()
orphan_ids_enroll = set(df_enrollments['student_id']) - set(df_students['student_id'])
orphan_ids_activity = set(df_activity['student_id']) - set(df_students['student_id'])

print(f"Final roster row count: {len(df_integrated)}")
print(f"Matched rows in enrollments: {matched_rows}")
print(f"Orphan IDs in enrollments: {len(orphan_ids_enroll)}")
print(f"Orphan IDs in weekly activity: {len(orphan_ids_activity)}")

Final roster row count: 1990
Matched rows in enrollments: 1990
Orphan IDs in enrollments: 24
Orphan IDs in weekly activity: 10


In [22]:
df_integrated['enrollment_date'] = pd.to_datetime(df_integrated['enrollment_date'], errors='coerce')
blanks_after_parse = df_integrated['enrollment_date'].isnull().sum()
print(f"Blank/invalid dates after parsing: {blanks_after_parse}")

mean_study = df_integrated['study_hours_reported'].mean()
std_study = df_integrated['study_hours_reported'].std()
df_integrated['study_hours_zscore'] = (df_integrated['study_hours_reported'] - mean_study) / std_study

gpa_bins = [0.0, 2.0, 2.5, 3.0, 3.5, 4.0]
gpa_labels = ['F/D', 'C', 'B-', 'B+/A-', 'A']
df_integrated['gpa_band'] = pd.cut(df_integrated['final_gpa'], bins=gpa_bins, labels=gpa_labels, include_lowest=True)

Blank/invalid dates after parsing: 1443


In [23]:
df_integrated.to_csv('cleaned_northgate_students.csv', index=False)
print("Successfully exported cleaned data to 'cleaned_northgate_students.csv'.")

mean_gpa = df_integrated['final_gpa'].mean()
print(f"Mean GPA (Cleaned Data): {mean_gpa:.2f}")

gpa_study_corr = df_integrated['final_gpa'].corr(df_integrated['study_hours_reported'])
print(f"Correlation between Study Hours and Final GPA: {gpa_study_corr:.2f}")

Successfully exported cleaned data to 'cleaned_northgate_students.csv'.
Mean GPA (Cleaned Data): 2.31
Correlation between Study Hours and Final GPA: 0.62
